In [2]:
library(survival)
library(fixest)

In [1]:
data=read.csv('paper_fixed_data.csv')

In [3]:
head(data)

,X,id,year,dis,cluster_id,role,Code,gender,yfp,npub,F,L,M,ST
,<int>,<int>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<chr>
1,0,29768070,2008,51,20009795,F,JPN,F,2008,5,2,0,0,Advanced
2,1,31240446,2008,51,20009795,F,JPN,F,2008,5,2,0,0,Advanced
3,2,29768070,2008,51,29411586,M,JPN,M,2000,74,3,0,5,Advanced
4,3,31240446,2008,51,29411586,M,JPN,M,2000,74,3,0,5,Advanced
5,4,43269102,2015,51,29411586,M,JPN,M,2000,74,3,0,5,Advanced
6,5,45225060,2016,58,29411586,M,JPN,M,2000,74,3,0,5,Advanced


In [4]:
data$npub=log(data$npub)

In [5]:
data$fdep=ifelse(data$role == 'F', 1, 0)
data$ldep=ifelse(data$role == 'L', 1, 0)

In [6]:
unique_ids=unique(data$id)

In [7]:
sample_id=sample(unique_ids,50000)
sample_data=data[data$id %in% sample_id,]

In [12]:
head(sample_data)

,X,id,year,dis,cluster_id,role,Code,gender,yfp,npub,F,L,M,ST,fdep,ldep
,<int>,<int>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<int>,<dbl>,<int>,<int>,<int>,<chr>,<dbl>,<dbl>
54,53,50731738,2018,31,20074220,M,JPN,M,1998,4.736198,0,3,35,Advanced,0,0
89,88,56912460,2015,96,8007915,M,JPN,F,2000,4.394449,0,0,20,Advanced,0,0
126,125,65796311,2009,100,67763529,M,JPN,M,2000,3.688879,2,0,17,Advanced,0,0
163,162,65796311,2009,100,6695961,M,JPN,M,1981,4.060443,0,8,30,Advanced,0,0
205,204,77866273,2012,82,34638262,M,JPN,M,1980,4.890349,0,13,14,Advanced,0,0
220,219,64609598,2019,29,65582094,M,JPN,M,1980,3.912023,2,6,18,Advanced,0,0


In [8]:
model <- feglm(fdep~gender+npub+ST| id,data = sample_data,family = binomial())
summary(model)

NOTE: 11,655 fixed-effects (93,746 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: fdep
Observations: 310,971 
Fixed-effects: id: 38,345
Standard-errors: Clustered (id) 
           Estimate Std. Error  t value  Pr(>|t|)    
genderM   -0.279762   0.018388 -15.2145 < 2.2e-16 ***
npub      -0.252556   0.005502 -45.8983 < 2.2e-16 ***
STLagging  0.489907   0.017488  28.0132 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -121,061.5   Adj. Pseudo R2: -0.136533
           BIC:  727,127.7     Squared Cor.:  0.125791

In [9]:
model <- feglm(ldep~gender+npub+ST| id,data = sample_data,family = binomial())
summary(model)

NOTE: 14,038 fixed-effects (111,399 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: ldep
Observations: 293,318 
Fixed-effects: id: 35,962
Standard-errors: Clustered (id) 
           Estimate Std. Error  t value  Pr(>|t|)    
genderM    0.290643   0.022368  12.9935 < 2.2e-16 ***
npub       0.613647   0.007813  78.5446 < 2.2e-16 ***
STLagging -0.283449   0.020811 -13.6203 < 2.2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
Log-Likelihood: -110,870.5   Adj. Pseudo R2: -0.069584
           BIC:  674,504.9     Squared Cor.:  0.217421

In [21]:
fitclogit=clogit(ldep~gender+npub+ST+strata(id),data = sample_data)

In [22]:
summary(fitclogit)

Call:
coxph(formula = Surv(rep(1, 408737L), ldep) ~ gender + npub + 
    ST + strata(id), data = sample_data, method = "exact")

  n= 408737, number of events= 52977 

               coef exp(coef)  se(coef)      z Pr(>|z|)    
genderM    0.222473  1.249162  0.013907  16.00   <2e-16 ***
npub       0.509175  1.663917  0.004578 111.23   <2e-16 ***
STLagging -0.213333  0.807887  0.012385 -17.23   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

          exp(coef) exp(-coef) lower .95 upper .95
genderM      1.2492     0.8005    1.2156    1.2837
npub         1.6639     0.6010    1.6491    1.6789
STLagging    0.8079     1.2378    0.7885    0.8277

Concordance= 0.709  (se = 0.003 )
Likelihood ratio test= 18700  on 3 df,   p=<2e-16
Wald test            = 14648  on 3 df,   p=<2e-16
Score (logrank) test = 16839  on 3 df,   p=<2e-16
